# Corpus gap audit: missing falsifiers, unstated conditions, and the questions left open

**Claim cluster:** `C075`, `C076`, `C077`, `C078`, `C088`, `C089`, `C090`, `C091`  
**Source of truth:** `data/smarthome.duckdb` (read-only), via `src/viz_core.py`

In [ ]:
import sys
from pathlib import Path

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, clear_output, display

# Find the repo root by walking up until schemas/stage2.sql appears, so the
# notebook runs from notebooks/, from the repo root, or from anywhere else.
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "schemas" / "stage2.sql").exists():
        ROOT = _candidate
        break
else:  # pragma: no cover - only reachable outside a checkout
    raise FileNotFoundError("schemas/stage2.sql not found above " + str(_here))

if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import viz_core as viz  # noqa: E402

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

# Renderers matter for two reasons. "browser" (plotly's default) opens a
# window when a figure is shown outside a live kernel, which breaks headless
# execution. The bare "notebook" renderer in plotly 7 emits only text/html,
# carrying no figure payload, so a notebook could run clean and still show
# nothing. "plotly_mimetype+notebook" emits the figure JSON for JupyterLab and
# an inline HTML fallback elsewhere -- which is also what the headless test
# asserts on.
pio.renderers.default = "plotly_mimetype+notebook"

# Every figure in these notebooks reads only from data/smarthome.duckdb.
CLAIM_IDS = ('C075', 'C076', 'C077', 'C078', 'C088', 'C089', 'C090', 'C091')
assert viz.DEFAULT_DB_PATH.endswith("smarthome.duckdb"), viz.DEFAULT_DB_PATH

claims = viz.fetch_claims(list(CLAIM_IDS))
sources = viz.sources_for_claims(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
cluster_metrics = (
    viz.claim_metrics_lookup(list(CLAIM_IDS)) if CLAIM_IDS else pd.DataFrame()
)

# Corpus-wide row counts, read once so a notebook can state the size of what it
# is drawing from without confusing a whole-corpus count with the size of its
# own (possibly empty) claim cluster.
CORPUS = viz.connect_db()
try:
    corpus_counts = {
        table: CORPUS.execute(f"SELECT COUNT(1) FROM {table}").fetchone()[0]
        for table in ("stage2_claim", "metric", "entity", "source_mirror",
                      "predicate", "extraction_decision")
    }
finally:
    CORPUS.close()

def provenance(claim_id):
    """Collapsible evidence panel for one claim in this cluster."""
    panel = viz.render_provenance_panel(claim_id)
    return widgets.VBox([panel])

def show(*items):
    """Display data frames, HTML, or figures, one per argument."""
    for item in items:
        display(item)

def confidence_legend():
    """Show the colour coding applied to confidence across all figures."""
    display(HTML(
        "<b>Confidence colour coding:</b> "
        f"<span style='color:{viz.CONFIDENCE_COLORS['high']}'>high</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['medium']}'>medium</span> &middot; "
        f"<span style='color:{viz.CONFIDENCE_COLORS['low']}'>low</span>"
    ))

## Provenance

This is the meta notebook of the catalogue: its subject is the corpus itself. The cluster is the set of claims in which the corpus states what it does not claim (`C088`-`C091`, all documented facts) and what it recommends next (`C075`-`C078`, all recommendations). Four of the eight carry no falsifier, which is the point of the page.

In [ ]:
for claim_id in CLAIM_IDS:
    display(HTML(f"<h4>{claim_id} &mdash; {claims.set_index('local_id').loc[claim_id, 'claim_text']}</h4>"))
    display(provenance(claim_id))
confidence_legend()

## Primary evidence

### The corpus's own gap flags, counted from the schema's views

Three views in `schemas/stage2.sql` expose the gaps directly: `missing_falsifiers`, `unstated_conditions` and `unstated_boundaries`. The fourth bar counts rows in `stage2_warning`, the ingestion-warning table. Nothing on this chart is estimated: each bar is a `COUNT(1)` over a view or table.

In [ ]:
# Every count below is a COUNT(1) against a schema view or table, read in one
# connection so the numbers are from a single point in time.
_con = viz.connect_db()
try:
    gap_counts = pd.DataFrame([
        {"gap": "claims with falsifier_stated = false",
         "source": "missing_falsifiers",
         "count": _con.execute("SELECT COUNT(1) FROM missing_falsifiers").fetchone()[0]},
        {"gap": "metrics with conditions_stated = false",
         "source": "unstated_conditions",
         "count": _con.execute("SELECT COUNT(1) FROM unstated_conditions").fetchone()[0]},
        {"gap": "entities with boundary_stated = false",
         "source": "unstated_boundaries",
         "count": _con.execute("SELECT COUNT(1) FROM unstated_boundaries").fetchone()[0]},
        {"gap": "ingestion warnings recorded",
         "source": "stage2_warning",
         "count": _con.execute("SELECT COUNT(1) FROM stage2_warning").fetchone()[0]},
    ])
    decisions = pd.DataFrame(
        _con.execute(
            "SELECT decision_type, COUNT(1) AS decisions FROM extraction_decision "
            "GROUP BY decision_type ORDER BY decisions DESC, decision_type"
        ).fetchall(),
        columns=["decision_type", "decisions"],
    )
    open_rows = pd.DataFrame(
        _con.execute(
            "SELECT decision_type, COUNT(1) AS rows FROM open_questions "
            "GROUP BY decision_type ORDER BY rows DESC"
        ).fetchall(),
        columns=["decision_type", "open_view_rows"],
    )
    stages = pd.DataFrame(
        _con.execute(
            "SELECT local_id, stage_name FROM workflow_stage ORDER BY local_id"
        ).fetchall(),
        columns=["stage", "stage_name"],
    )
finally:
    _con.close()

show(gap_counts)
fig = go.Figure(go.Bar(
    x=gap_counts["count"],
    y=gap_counts["gap"] + " (" + gap_counts["source"] + ")",
    orientation="h",
    marker_color="#b06000",
    text=gap_counts["count"],
    textposition="outside",
    hovertemplate="%{y}<br>%{x} rows<extra></extra>",
))
fig.update_layout(
    title=("Recorded gaps, straight from the schema views "
           "(no gap is estimated from text)"),
    xaxis_title="rows",
    height=320,
    yaxis=dict(automargin=True, autorange="reversed"),
)
fig.show()

display(HTML(
    "<h4>What the <code>open_questions</code> view actually returns</h4>"
    "<p>The view unions two decision types, so its own contents are shown "
    "before the full decision profile.</p>"
))
show(open_rows)
show(decisions)
_open_types = {"open_question", "omission"}
fig = go.Figure(go.Bar(
    x=decisions["decisions"],
    y=decisions["decision_type"],
    orientation="h",
    marker_color=["#b32637" if t in _open_types else "#4c78a8"
                  for t in decisions["decision_type"]],
    text=decisions["decisions"],
    textposition="outside",
    hovertemplate="%{y}<br>%{x} decisions<extra></extra>",
))
fig.update_layout(
    title=("Every extraction decision type, counted (red = the two types the "
           "open_questions view exposes)"),
    xaxis_title="decisions",
    height=420,
    yaxis=dict(automargin=True, autorange="reversed"),
)
fig.show()

## Supporting context

### Where the corpus's claims sit in its own workflow

Every claim carries a workflow stage (`W3`-`W12`). Counting claims per stage is the most direct measure of where this corpus spent its research effort, and the stage names come from `workflow_stage` rather than from this notebook. The second table is what the corpus proposes to do about the gaps: the mixed-method study design behind stage `W10` (`M145`-`M152`), which is a plan rather than a result.

In [ ]:
_corpus = viz.fetch_all_claims()
_stage_counts = (_corpus.groupby("workflow_stage").size()
                 .reset_index(name="claims"))
_stage_view = stages.merge(_stage_counts, left_on="stage",
                           right_on="workflow_stage", how="left")
_stage_view["claims"] = _stage_view["claims"].fillna(0).astype(int)
_stage_view = _stage_view.sort_values("stage")
show(_stage_view.rename(columns={"stage_name": "name"}))

fig = go.Figure(go.Bar(
    x=_stage_view["claims"],
    y=_stage_view["stage"] + " " + _stage_view["stage_name"],
    orientation="h",
    marker_color="#4c78a8",
    text=_stage_view["claims"],
    textposition="outside",
    hovertemplate="%{y}<br>%{x} claims<extra></extra>",
))
fig.update_layout(
    title=("Claims per workflow stage, with the stage names taken from the "
           "workflow_stage table"),
    xaxis_title="claims",
    height=470,
    yaxis=dict(automargin=True, autorange="reversed"),
)
fig.show()

display(HTML(
    "<h4>The follow-up study design the corpus proposes (M145-M152)</h4>"
    "<p>These rows describe a study that has not been run: sample sizes, "
    "instrument design, incentives, cost and the corpus's own quality "
    "self-assessment.</p>"
))
show(viz.metrics_by_id(["M145", "M146", "M147", "M148", "M149", "M150",
                        "M151", "M152"])[
    ["local_id", "metric_name", "value", "unit", "confidence", "source_ref",
     "section"]].rename(
    columns={"local_id": "metric", "metric_name": "name",
             "source_ref": "source", "confidence": "conf"}))
display(HTML(
    "<h4>Claim cluster evidence table</h4>"
    "<p>Every claim in this cluster, with its evidence class, confidence, "
    "workflow stage and cited source ids. This is the table the figures above "
    "are built from.</p>"
))
show(
    claims[["local_id", "claim_type", "evidence_marker", "confidence",
            "workflow_stage", "source_refs", "falsifier_missing", "section"]]
    .rename(columns={
        "local_id": "claim", "claim_type": "type", "evidence_marker": "ev",
        "confidence": "conf", "workflow_stage": "stage",
        "source_refs": "sources", "falsifier_missing": "no_falsifier",
    })
)

display(HTML("<h4>Sources cited by the cluster</h4>"))
show(sources[["local_id", "title", "publisher", "classification",
              "publication_date", "url"]].rename(
    columns={"local_id": "source"}))

## Uncertainty and gaps

### These counts measure flags, not quality

A claim without a falsifier is not automatically a bad claim: `C001`-`C005` are documented facts about a specification, and some claims are recommendations (`C075`-`C078`) for which a falsifier is not the right instrument. What the counts measure is the corpus's *discipline in recording disconfirming observations*, which is what the extraction log tracks.

Other limits on this page:

- **The ingestion-warning table is empty.** `stage2_warning` holds no rows in this database, so the gap mechanism that is actually populated is `extraction_decision` (56 rows across 11 types).
- **The `open_questions` view is a union, not a backlog.** It returns `open_question` and `omission` decisions together, and most of its rows are omissions recorded during ingestion rather than research questions.
- **`C088`-`C091` are the corpus's own caution.** Four documented facts state that the research does not prove Matter is unreliable, does not prove universal energy savings, does not establish security rankings as fact, and does not support company-level investment conclusions. They are why this page reports gaps rather than a quality score.
- **The study design is a proposal.** `M150` (285,000-340,000 USD over 11 months) and `M151` (quality scores out of 5) describe a plan; nothing in this database shows it was executed.
- **Recommendations carry no falsifiers by design.** `C075`-`C078` are `recommendation` claims and appear in the no-falsifier list, which is expected rather than a defect.

In [ ]:
# Two independent readings of 'no falsifier' are compared rather than merged:
# the stored boolean flag and the literal placeholder text. They agree in this
# database, and stating that out loud is worth more than assuming it.
_all = viz.fetch_all_claims()
_flag_missing = set(_all.loc[~_all["falsifier_stated"], "local_id"])
_text_missing = set(_all.loc[_all["falsifier_missing"], "local_id"])
_compare = pd.DataFrame([{
    "flag_says_missing": len(_flag_missing),
    "cell_text_says_missing": len(_text_missing),
    "disagreements": len(_flag_missing ^ _text_missing),
}])
display(HTML(
    "<h4>Flag against placeholder text</h4>"
    "<p>The view <code>missing_falsifiers</code> reads the stored flag; "
    "<code>viz_core</code> also treats the literal placeholder "
    "<code>[falsifier not stated]</code> as a missing falsifier. The two "
    "readings are compared here, not merged.</p>"
))
show(_compare)

display(HTML("<h4>Which claim types the no-falsifier claims are</h4>"))
show(_all.loc[~_all["falsifier_stated"]]
     .groupby(["claim_type", "confidence"]).size()
     .reset_index(name="claims")
     .rename(columns={"claim_type": "type", "confidence": "conf"}))

display(HTML("<h4>The claims themselves</h4>"))
show(_all.loc[~_all["falsifier_stated"],
              ["local_id", "claim_type", "confidence", "workflow_stage",
               "falsifier_missing"]].rename(
    columns={"local_id": "claim", "claim_type": "type", "confidence": "conf",
             "workflow_stage": "stage", "falsifier_missing": "no_falsifier"}))
display(HTML("<h4>Claims in this cluster with no falsifier</h4>"
             "<p>A missing falsifier means the corpus recorded no observation "
             "that would disconfirm the claim. The panel above flags these "
             "claims in <code>no_falsifier</code>.</p>"))
show(claims.loc[claims["falsifier_missing"], ["local_id", "claim_type",
                                              "confidence"]]
     .rename(columns={"local_id": "claim", "claim_type": "type",
                      "confidence": "conf"}))

display(HTML("<h4>Metrics with conditions not stated in the source</h4>"))
_gaps = cluster_metrics.loc[~cluster_metrics["conditions_stated"].fillna(False)]
if _gaps.empty:
    display(HTML("<p>No unstated conditions among the cluster's metrics.</p>"))
else:
    show(_gaps[["local_id", "metric_name", "value", "unit"]].rename(
        columns={"local_id": "metric", "metric_name": "name"}))

display(HTML("<h4>Extraction decisions that changed or qualified this material</h4>"))
_decisions = pd.DataFrame(
    viz.connect_db().execute(
        "SELECT local_id, decision_type, description, resolution "
        "FROM extraction_decision "
        "WHERE decision_type IN ('evidence_downgrade', 'classification_conflict', "
        "                           'falsifier_absent', 'condition_absent', "
        "                           'boundary_absent', 'scope_boundary', "
        "                           'omission') "
        "ORDER BY local_id"
    ).fetchall(),
    columns=["decision", "type", "description", "resolution"],
)
show(_decisions)

## Reproducibility

Gap counts are `COUNT(1)` queries against `missing_falsifiers`, `unstated_conditions`, `unstated_boundaries` and `stage2_warning`; the decision profile against `extraction_decision` and `open_questions`; stage names against `workflow_stage`. Claim-level counts come from `viz.fetch_all_claims`. Every number is computed at run time, so a re-import changes the page.

In [ ]:
# The exact queries behind every figure above, re-run verbatim.
import duckdb
con = duckdb.connect(str(ROOT / 'data' / 'smarthome.duckdb'), read_only=True)
print('database:', con.execute('SELECT current_database()').fetchone()[0])
for table in ('stage2_claim', 'metric', 'entity', 'source_mirror',
               'predicate', 'workflow_stage', 'extraction_decision'):
    total = con.execute(f'SELECT COUNT(*) FROM {table}').fetchone()[0]
    print(f'{table:22} {total:>6} rows')
con.close()